In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
import lightning as L
import torch.nn as nn
import torch
import torch.nn.functional as F
from torchmetrics import MeanAbsoluteError, MeanSquaredError
from torch.utils.data import DataLoader, Dataset

In [2]:
L.seed_everything(12)

Seed set to 12


12

In [3]:
movies_df = pd.read_csv("ml-latest-small/movies.csv")
ratings_df = pd.read_csv("ml-latest-small/ratings.csv")

In [4]:

"""Data preprocessing"""

movies_df["production_year"] = movies_df.title.str.extract(
    r"\((\d{4})\)", expand=False
).astype(float)  # extracting years from titles
movies_df["title"] = movies_df.title.str.replace(
    pat=r"\(\d{4}\)", repl="", regex=True
)  # deleteing years of production from titles
movies_df["genres"] = movies_df.genres.str.split("|")

print(
    movies_df["production_year"].min(), movies_df["production_year"].max()
)  # range of production years

mlb = MultiLabelBinarizer()

movies_genres = pd.DataFrame(
    mlb.fit_transform(movies_df.genres), columns=mlb.classes_, index=movies_df.index
)

movies_df = pd.concat(
    [movies_df, movies_genres], axis=1
)  # join movies_df and movies_genres along columns

mms = MinMaxScaler(feature_range=(0, 1))


movies_df["years_scaled"] = mms.fit_transform(movies_df[["production_year"]])

data = ratings_df.merge(movies_df, on="movieId")

data["years_scaled"].fillna(
    data["years_scaled"].median(), inplace=True
)  # filling missing years with years median


movie_counts = data.groupby("movieId").size()  # get counts of movie ratings
movies_to_keep = movie_counts[
    movie_counts >= 3
].index  # get indices of movies with ratings >= 3

data = data[
    data["movieId"].isin(movies_to_keep)
]  # only leaves rows with movieId which are in movies_to_keep


def user_train_test_split(df, train_size=0.8, seed=12):
    np.random.seed(seed)

    train_list = []
    test_list = []

    for userId, user_df in df.groupby("userId"):
        number_of_ratings = len(user_df)
        ratings_number_to_train = int(number_of_ratings * train_size)

        train_indices = np.random.choice(
            user_df.index, size=ratings_number_to_train, replace=False
        )
        test_indices = user_df.index.difference(train_indices)

        train_list.append(df.loc[train_indices])
        test_list.append(df.loc[test_indices])

    train_df = pd.concat(train_list, axis=0)
    test_df = pd.concat(test_list, axis=0)

    user2idx = {user: i for i, user in enumerate(df["userId"].unique())}
    movie2idx = {movie: i for i, movie in enumerate(df["movieId"].unique())}

    # Apply mappings to train and test, so each original userId is replaced with a 0-based integer
    train_df["userId_cat"] = train_df["userId"].map(user2idx)
    train_df["movieId_cat"] = train_df["movieId"].map(movie2idx)
    test_df["userId_cat"] = test_df["userId"].map(user2idx)
    test_df["movieId_cat"] = test_df["movieId"].map(movie2idx)

    return train_df, test_df


train_df, test_df = user_train_test_split(data)

features_list = [
    "userId_cat",
    "movieId_cat",
    "years_scaled",
] + mlb.classes_.tolist()


/tmp/ipykernel_1778726/4009331880.py:32: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["years_scaled"].fillna(


1902.0 2018.0


In [5]:
class MovieDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X.values).float()
        self.y = torch.from_numpy(y.values).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


In [6]:
class RecommendationModel(nn.Module):
    def __init__(self,
        num_users,
        num_movies,
        embedding_dim,
        features_dim,
        out_channels,
        hidden_channels):
        super().__init__()

        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.movie_embedding = nn.Embedding(num_movies, embedding_dim)

        in_channels = 2 * embedding_dim + features_dim  # 2 * coz 2x embedding

        self.net = nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_channels // 2, out_channels),
        )

       
    def forward(self, user_ids, movie_ids, features):
        user_emb = self.user_embedding(user_ids)
        movie_emb = self.movie_embedding(movie_ids)

        x = torch.cat([user_emb, movie_emb, features], dim=1)

        return self.net(x).squeeze(-1)

In [ ]:
class RecommenderLightingModule(L.LightningModule):
    def __init__(self, num_users,
        num_movies,
        embedding_dim,
        features_dim,
        out_channels,
        hidden_channels):
        super().__init__()
        
        self.recommender = RecommendationModel(num_users,
        num_movies,
        embedding_dim,
        features_dim,
        out_channels,
        hidden_channels)
        self.rmse = MeanSquaredError(squared=False)
        self.mae = MeanAbsoluteError()

    def training_step(self, batch):
        X, y = batch
        user_ids = X[:, 0].long()
        movie_ids = X[:, 1].long()
        side_features = X[:, 2:]

        logits = self.recommender(user_ids, movie_ids, side_features)
        loss = F.mse_loss(logits, y)
        mae = self.mae(logits, y)
        rmse = self.rmse(logits, y)
        
        self.log("train/loss", loss)
        self.log("train/mae", mae)
        self.log("train/rmse", rmse)
        return loss

    def validation_step(self, batch):
        X, y = batch

        user_ids = X[:, 0].long()
        movie_ids = X[:, 1].long()
        side_features = X[:, 2:]

        logits = self.recommender(user_ids, movie_ids, side_features)
        mae = self.mae(logits, y)
        rmse = self.rmse(logits, y)

        self.log("val/mae", mae)
        self.log("val/rmse", rmse)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters())
        

In [8]:
recommender = RecommenderLightingModule(num_users=data.userId.nunique(),
    num_movies=data.movieId.nunique(),
    embedding_dim=16,
    features_dim=len(features_list) - 2,
    out_channels=1,
    hidden_channels=64,)

In [9]:
batch_size = 32

trainset = MovieDataset(train_df[features_list], data["rating"])
testset = MovieDataset(test_df[features_list], data["rating"])

train_loader = DataLoader(trainset, batch_size, shuffle=True, num_workers=8)
test_loader = DataLoader(testset, batch_size, shuffle=False, num_workers=8)

In [10]:
trainer = L.Trainer(max_epochs=30)
trainer.fit(recommender, train_loader, test_loader)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name        | Type                | Params | Mode 
------------------------------------------------------------
0 | recommender | RecommendationModel | 95.1 K | train
1 | rmse        | MeanSquaredError    | 0      | train
2 | mae         | MeanAbsoluteError   | 0      | train
------------------------------------------------------------
95.1 K    Trainable params
0         Non-trainable params
95.1 K    Total params
0.381     Total estimated model params size (MB)
14        Modules in train mode
0         Modules in eval mode


Epoch 10:  59%|█████▉    | 1400/2363 [00:09<00:06, 154.79it/s, v_num=2]    


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/emykowska/projects/movies_recommendations/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
